In [94]:
import pandas as pd

In [62]:
!pip install pyspark

In [95]:
# Step 2: Start Spark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count,avg, min, max

In [162]:
# Step 3: Load Data

spark = SparkSession.builder.appName("CSV Loader").getOrCreate()
df = spark.read.csv(r"C:\Users\HP\Downloads\Superstore.csv",header=True,inferSchema=True)

In [147]:
# 3.First few rows
df.show(8)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [148]:
# 3. Column names
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [149]:
# 3. Data types
df.dtypes

[('Row ID', 'string'),
 ('Order ID', 'string'),
 ('Order Date', 'string'),
 ('Ship Date', 'string'),
 ('Ship Mode', 'string'),
 ('Customer ID', 'string'),
 ('Customer Name', 'string'),
 ('Segment', 'string'),
 ('Country', 'string'),
 ('City', 'string'),
 ('State', 'string'),
 ('Postal Code', 'string'),
 ('Region', 'string'),
 ('Product ID', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('Sales', 'string'),
 ('Quantity', 'string'),
 ('Discount', 'string'),
 ('Profit', 'string')]

In [150]:
# Step 4: Data Cleaning

df.count()-df.distinct().count()

0

In [151]:
df = df.dropDuplicates()

In [152]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [153]:
# Step 5: Filter Data
# (Filter by region)
df.filter(df["Region"] == "West").show()

+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|      Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|   144|CA-2017-106180| 9/18/2017| 9/23/2017|Standard Class|   SH-19975|      Sally Hughsby|  Corporate|United States|San Francisco|California|      94122|  West|OFF-PA-10004327|Office Supplies|       Paper| 

In [154]:
# (Filter by category)
df.filter(df["Category"] == "Furniture").show()

+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|  Segment|      Country|         City|       State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|   229|US-2015-145436| 2/28/2015|  3/4/2015|Standard Class|   VD-21670|Valerie Dominguez| Consumer|United States|     Columbia|   Tennessee|      38401|  South|FUR-CH-10004860|Furniture|      Chairs|Global Low Back T...|161.5

In [105]:
# (Filter by Profit)
df.filter(df["Profit"] > 100).show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|  4813|CA-2016-121370|11/14/2016|11/19/2016|  Second Class|   EB-14110|   Eugene Barchas|   Consumer|United States|   Philadelphia|Pennsylvania|      19134|   East|TEC-CO-10004115|     Technology

In [106]:
# Step 6: Transform Data
df = df.withColumn("Sales", col("Sales").cast("float"))
df = df.withColumn("Discount", col("Discount").try_cast("float"))
df = df.withColumn("Quantity", col("Quantity").try_cast("float"))

In [107]:
df = df.withColumnRenamed("Ship Mode", "Shipment Mode")
df = df.withColumnRenamed("Ship Date", "Shipment Date")

In [108]:
df.dtypes

[('Row ID', 'int'),
 ('Order ID', 'string'),
 ('Order Date', 'string'),
 ('Shipment Date', 'string'),
 ('Shipment Mode', 'string'),
 ('Customer ID', 'string'),
 ('Customer Name', 'string'),
 ('Segment', 'string'),
 ('Country', 'string'),
 ('City', 'string'),
 ('State', 'string'),
 ('Postal Code', 'int'),
 ('Region', 'string'),
 ('Product ID', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('Sales', 'float'),
 ('Quantity', 'float'),
 ('Discount', 'float'),
 ('Profit', 'double')]

In [155]:
# Step 7: Aggregation
# (total rows)
print("Total Rows:", df.count())

Total Rows: 9994


In [156]:
# (average values)
df.select(
    avg("Quantity").alias("Average Quantity"),
          avg("Discount").alias("Average Discount"),
          avg("Profit").alias("Average Profit")).show()

+-----------------+-------------------+-----------------+
| Average Quantity|   Average Discount|   Average Profit|
+-----------------+-------------------+-----------------+
|3.789573744246548|0.15620272163298343|28.65689630778464|
+-----------------+-------------------+-----------------+



In [157]:
# (minimum and maximum values)
df.select(
    min("Quantity").alias("Min Quantity"),
    max("Quantity").alias("Max Quantity"),
    min("Discount").alias("Min Discount"),
    max("Discount").alias("Max Discount"),
    min("Profit").alias("Min Profit"),
    max("Profit").alias("Max Profit")
).show()

+------------+------------+------------+------------+----------+----------+
|Min Quantity|Max Quantity|Min Discount|Max Discount|Min Profit|Max Profit|
+------------+------------+------------+------------+----------+----------+
|           1|           9|           0|         0.8|   -0.0895|   99.9408|
+------------+------------+------------+------------+----------+----------+



In [158]:
# Step 8: Group Data 
df.groupBy("Category").count().show()

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies| 6026|
|      Furniture| 2121|
|     Technology| 1847|
+---------------+-----+



In [159]:
df.groupBy("Category").agg(sum("Quantity").alias("Total Quantity")).show()

+---------------+--------------+
|       Category|Total Quantity|
+---------------+--------------+
|Office Supplies|       22906.0|
|      Furniture|        8028.0|
|     Technology|        6939.0|
+---------------+--------------+



In [91]:
df.groupBy("Region").agg(avg("Profit").alias("Average Profit")).show()

+-------+------------------+
| Region|    Average Profit|
+-------+------------------+
|  South|28.796506790123438|
|Central| 17.28390142057685|
|   East|32.163994627809004|
|   West| 33.50099953168906|
+-------+------------------+



In [161]:
df.toPandas().to_csv(
    r"C:\Users\HP\Downloads\results.csv",
    index=False
)